# SSMP-ZeroShot: A Simplified Zero-Shot Recreation of
# "Adaptive Self-Supporting Prototype Learning for Remote Sensing Few-Shot Semantic Segmentation"

**Paper:** Weihao Shen, Ailong Ma, Junjue Wang, Zhuo Zheng, Yanfei Zhong,
*"Adaptive Self-Supporting Prototype Learning for Remote Sensing Few-Shot Semantic Segmentation"*,
IEEE Transactions on Geosciences and Remote Sensing, vol. 62, 2024.
DOI: `10.1109/TGRS.2024.3435086`

---

## What this notebook is

This is a **simplified zero-shot adaptation / recreation** of the paper's **SSMP** framework
for a B.Tech-level research project. It is **not** an exact reproduction of the authors' few-shot
implementation, and it is **not** a generic unrelated CLIP segmentation project: every major
module of the paper is present in the code, re-wired from *few-shot* to *zero-shot*.

### Original paper (few-shot)

```text
Few-shot semantic segmentation
Support images + masks
      |
      v
Prototype learning        (masked average pooling of support features)
      |
      v
HPR  = HPC + GPM          (multiple hyperprototypes + guided allocation)
      |
      v
SSM                        (pseudo labels -> query prototypes -> re-match)
      |
      v
ASBP                       (adaptive self-supporting background prototype)
      |
      v
segmentation  (evaluated with mIoU / FB-IoU)
```

### Our recreation (zero-shot)

```text
Zero-shot semantic segmentation
Class names / text prompts
      |
      v
CLIP semantic prototypes   (replaces "support images -> visual prototypes")
      |
      v
HPR-inspired hyperprototypes   (simplified HPC + GPM)
      |
      v
pixel-prototype matching       (cosine similarity, GPM)
      |
      v
SSM-inspired self-supporting refinement   (pseudo mask -> query prototypes)
      |
      v
ASBP-inspired background prototype        (adaptive per-pixel background)
      |
      v
segmentation  (evaluated with mIoU / FB-IoU)
```

### Why the change is needed

In the original few-shot setting the class prototype is computed from **k support images and
their masks**. In the **zero-shot** setting **there is no support image and no support mask at
all**, so the initial class prototype must come from *language*:

```text
Original paper:   Support images  -> visual prototypes
Our zero-shot:     Class descriptions -> CLIP text prototypes
```

Everything downstream (HPR, GPM, SSM, ASBP, prototype refinement, pixel-level similarity
matching, mIoU evaluation) is kept, adapted so that it works with text-derived prototypes.

> **Disclaimer.** This is a simplified, conceptually similar recreation. Some of the paper's
> mathematical details (SLIC superpixel seed growth, the exact affinity formulation, the
> support-branch losses, multi-episode meta-learning) are intentionally simplified so the whole
> pipeline runs end-to-end on a free Google Colab GPU. All metrics printed by this notebook come
> from actually running it - no numbers are pre-filled or fabricated.


## 2. Paper recreation vs. original paper - module-by-module mapping

| # | Paper module (few-shot) | This notebook (zero-shot) | What is simplified |
|---|---|---|---|
| 1 | Support images + masks | *none* - class names / prompts | No support set at all |
| 2 | Support prototypes via masked average pooling | **CLIP text prototypes** + learnable text adapter | Prototype comes from language |
| 3 | ResNet-50/101 frozen backbone, spatial feature map | `ResNetFeatureExtractor` (frozen ResNet-50, keeps layer3+layer4 maps) | Concatenates two stages |
| 4 | **HPR** = **HPC** + **GPM** | `SimplifiedHPR` + `gpm_match` | K-means in feature space replaces SLIC superpixel clustering |
| 5 | Hyperprototype clustering (parameter-free) | `kmeans_prototypes` (feature + spatial distance, few iterations) | No MaskSLIC seed growth |
| 6 | Guided prototype matching (argmax allocation) | `gpm_match` (argmax allocation + soft aggregation) | Soft aggregation added |
| 7 | **SSM** pseudo mask -> query prototypes | `confident_masks` + `masked_average` | Same idea, one iteration |
| 8 | **ASBP** affinity-matrix background prototypes | `asbp_background` (Eq. 9-10 style affinity + softmax) | Cap on background pixels |
| 9 | Weighted prototype fusion (lambda coefficients) | learnable `log_lambda` fused with softmax | Same idea, 2 weights |
| 10 | Prototype refinement ("refine" operation) | second pseudo-label round in `ZeroShotSSMP.forward` | One extra round |
| 11 | BCE losses `L = L_seg + beta * L_pseudo`, beta = 0.2 | Cross-entropy on `[background, foreground]` scores, same 0.2 weight | CE instead of BCE |
| 12 | mIoU + FB-IoU on iSAID-5i | mIoU + per-class IoU + pixel acc + Dice + FB-IoU | Extra diagnostics |
| 13 | ResNet frozen, only light layers trained | backbone + CLIP frozen; projection, text adapter, fusion weights trained | Same philosophy |

**Class split.** iSAID-5i divides the 15 iSAID classes into three *disjoint* folds of five
classes. This notebook uses one five-class fold as **unseen (zero-shot test) classes** and the
remaining ten as **base (training) classes**, defined explicitly in the Configuration cell so the
experiment is reproducible. If you have the official iSAID-5i fold lists, simply paste them into
`UNSEEN_CLASSES`.


In [ ]:
# =============================================================================
# 3. INSTALL DEPENDENCIES
# =============================================================================
# Colab already ships torch / torchvision / matplotlib / sklearn. We only make
# sure everything we need is importable, installing quietly if something is missing.
import importlib
import subprocess
import sys


def _ensure(pkg, import_name=None):
    """Install `pkg` with pip only when its import name is not available."""
    import_name = import_name or pkg
    try:
        importlib.import_module(import_name)
    except ImportError:
        print(f"Installing {pkg} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])


for _pkg, _name in [
    ("torchvision", "torchvision"),
    ("transformers", "transformers"),
    ("scikit-learn", "sklearn"),
    ("matplotlib", "matplotlib"),
    ("pandas", "pandas"),
    ("Pillow", "PIL"),
    ("numpy", "numpy"),
]:
    _ensure(_pkg, _name)

print("All required packages are available.")


In [ ]:
# =============================================================================
# 4. IMPORTS
# =============================================================================
import os
import json
import math
import time
import random
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import torchvision
from torchvision import transforms
from torchvision.models import ResNet50_Weights

from transformers import CLIPModel, CLIPProcessor

from sklearn.manifold import TSNE

print("torch        :", torch.__version__)
print("torchvision  :", torchvision.__version__)
print("transformers :", __import__("transformers").__version__)
print("numpy        :", np.__version__)


In [ ]:
# =============================================================================
# 5. CONFIGURATION
# =============================================================================
SEED = 42
def set_seed(seed=SEED):
    """Fix every random generator so repeated runs are reproducible."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
set_seed(SEED)
# ---------------------------------------------------------------- hardware --
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: running on CPU - training will be slow. Enable a Colab GPU (Runtime > Change runtime type).")
# ------------------------------------------------------------ image sizes ---
IMG_SIZE = 256           # the paper crops iSAID images to 256 x 256
EMBED_DIM = 512          # CLIP ViT-B/32 text embedding dimension
FEAT_SIZE = IMG_SIZE // 16   # ResNet stride 16 -> 16 x 16 spatial feature map
SCORE_SCALE = 8.0        # temperature applied to cosine scores before softmax
# ------------------------------------------------------------- iSAID class --
# 15 iSAID classes; mask value 0 = background, 1..15 = the classes below.
ISAID_CLASSES = [
    "plane", "ship", "storage tank", "baseball diamond", "tennis court",
    "basketball court", "ground track field", "soccer ball field", "harbor",
    "bridge", "large vehicle", "small vehicle", "helicopter", "roundabout",
    "swimming pool",
]
CLASS_TO_ID = {name: i + 1 for i, name in enumerate(ISAID_CLASSES)}
ID_TO_CLASS = {i: name for name, i in CLASS_TO_ID.items()}
# -------------------------------------------------- base / unseen split -----
# iSAID-5i splits the 15 classes into three disjoint folds of five classes.
# We use one five-class fold as UNSEEN (zero-shot) classes; the other ten are
# BASE classes used for training. The split is written out explicitly so the
# experiment is fully reproducible; replace UNSEEN_CLASSES if you have the
# official fold lists.
UNSEEN_CLASSES = ["ship", "plane", "harbor", "bridge", "large vehicle"]
BASE_CLASSES = [c for c in ISAID_CLASSES if c not in UNSEEN_CLASSES]
BASE_IDS = sorted(CLASS_TO_ID[c] for c in BASE_CLASSES)
UNSEEN_IDS = sorted(CLASS_TO_ID[c] for c in UNSEEN_CLASSES)
assert set(BASE_IDS).isdisjoint(UNSEEN_IDS), "base and unseen classes must be disjoint"
assert len(BASE_CLASSES) + len(UNSEEN_CLASSES) == 15
# ------------------------------------------------------------ data paths ----
# Expected layout (see the dataset section below):
#   /content/data/iSAID5i/images/*.png|jpg      RGB patches
#   /content/data/iSAID5i/masks/*.png           single-channel class ids 0..15
#   /content/data/iSAID5i/splits/train.txt      file stems (optional)
DATA_ROOT = Path("/content/data/iSAID5i")
DATA_ROOT_CANDIDATES = [
    Path("/content/data/iSAID5i"),
    Path("/content/iSAID5i"),
    Path("/content/drive/MyDrive/iSAID5i"),
]
ALLOW_SYNTHETIC_FALLBACK = True   # False -> fail hard when real data is missing
MAX_SCAN_IMAGES = 4000            # how many masks are inspected while filtering
# ------------------------------------------------- subset size (Colab-friendly)
MAX_TRAIN_IMAGES = 500
MAX_VAL_IMAGES = 80
MAX_TEST_IMAGES = 40              # per unseen class for zero-shot evaluation
ABLATION_IMAGES_PER_CLASS = 10    # smaller budget for the ablation study
# ------------------------------------------------------------- training -----
EPOCHS = 25
MIN_LR = 1e-5
BATCH_SIZE = 8
LR = 5e-4
WEIGHT_DECAY = 1e-4
USE_PSEUDO_LOSS = True
PSEUDO_LOSS_WEIGHT = 0.2          # beta in the paper: L = L_seg + beta * L_pseudo
FG_LOSS_WEIGHT = 3.0              # foreground class weight (objects are rare vs. background)
NUM_WORKERS = 2
# ------------------------------------------------------- model / modules ----
NUM_HYPERPROTOS = 4               # HPR: hyperprototypes generated per class
KMEANS_ITERS = 5                  # HPC-style clustering iterations
FG_THRESH = 0.6                   # SSM foreground confidence threshold
BG_THRESH = 0.6                   # SSM background confidence threshold
# switches of the FULL model (the ablation study changes these per config)
MODULES = {"hpr": True, "ssm": True, "asbp": True, "refine": True}
# ----------------------------------------------------------- checkpoints ----
CKPT_DIR = Path("/content/checkpoints")
CKPT_DIR.mkdir(parents=True, exist_ok=True)
BEST_CKPT = CKPT_DIR / "best_ssmp_zeroshot.pt"
# Optional: {(R,G,B): class_id} if your masks are colour-coded PNGs.
CLASS_COLOR_MAP = {}
print(f"BASE   classes ({len(BASE_CLASSES)}) : {BASE_CLASSES}")
print(f"UNSEEN classes ({len(UNSEEN_CLASSES)}) : {UNSEEN_CLASSES}")

## 6. Dataset download / loading - iSAID-5i

The paper trains and evaluates on **iSAID-5i**, a few-shot benchmark rebuilt from the large-scale
**iSAID** dataset (655,451 object instances, 15 categories, 2,806 high-resolution images). The
authors randomly cropped 256 x 256 patches, producing 18,076 training and 6,363 validation
images, and split the 15 classes into three disjoint 5-class folds.

### Where to get the data (official sources only - no invented URLs)

1. Official dataset page (contains the Google Drive / Baidu Drive download links):
   <https://captain-whu.github.io/iSAID/dataset.html>
2. Official development kit (patch splitting + preprocessing code):
   <https://github.com/CAPTAIN-WHU/iSAID_Devkit>
3. Original iSAID paper: <https://arxiv.org/abs/1905.12886>

> The full dataset is several GB. On Colab you only need a **manageable subset**: download the
> training + validation archives, keep a few hundred image/mask pairs, resize them to 256 x 256,
> and place them in the layout below. The notebook also accepts a copy in Google Drive.

### Directory layout this notebook expects

```text
iSAID5i/
├── images/                 RGB patches  (*.png | *.jpg), any size (resized to 256x256)
├── masks/                  semantic masks with the SAME file stem
│                           values = class ids: 0 = background, 1..15 = iSAID classes
└── splits/                 (optional)
    ├── train.txt           one file stem per line
    ├── val.txt
    └── test.txt
```

If `splits/*.txt` is missing, the notebook creates a deterministic 70/15/15 split.
If masks are **colour-coded** instead of single-channel ids, fill `CLASS_COLOR_MAP` in the
Configuration cell with `{(R, G, B): class_id}`.

### Fallback

If no dataset is found and `ALLOW_SYNTHETIC_FALLBACK = True`, a small **synthetic placeholder
dataset** (coloured shapes on noisy backgrounds, same directory layout, same 15 class names) is
generated so that every cell still runs end-to-end. **Those images are NOT remote sensing data
and any metric computed on them is meaningless** - a loud warning is printed whenever it is used.
Set `ALLOW_SYNTHETIC_FALLBACK = False` to force real data only.


In [ ]:
# =============================================================================
# 6b. LOCATE THE DATASET (with error handling) + OPTIONAL SYNTHETIC FALLBACK
# =============================================================================
import colorsys

IMG_EXTS = (".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff")
IMG_DIR_NAMES = {"images", "img", "image", "jpegimages"}
MASK_DIR_NAMES = {"masks", "mask", "semantic_masks", "semanticmask", "gt", "labels"}


def find_dirs(root, wanted_names, max_depth=3):
    """Return {dirname: [Path, ...]} for directories whose name is in wanted_names."""
    wanted = {w.lower() for w in wanted_names}
    found = {}
    root = Path(root)
    if not root.exists():
        return found
    for dirpath, dirnames, _ in os.walk(root):
        depth = len(Path(dirpath).relative_to(root).parts)
        if depth > max_depth:
            dirnames.clear()
            continue
        name = Path(dirpath).name.lower()
        if name in wanted:
            found.setdefault(name, []).append(Path(dirpath))
    return found


def locate_dataset(candidates):
    """Find the first candidate root that contains both an image and a mask dir."""
    for root in candidates:
        root = Path(root)
        imgs = find_dirs(root, IMG_DIR_NAMES)
        msks = find_dirs(root, MASK_DIR_NAMES)
        if imgs and msks:
            img_dir = sorted(imgs.values(), key=len)[0][0]
            mask_dir = sorted(msks.values(), key=len)[0][0]
            return {"root": root, "images": img_dir, "masks": mask_dir}
    return None


# ------------------------------------------------------- synthetic fallback --
def _class_color(cid):
    """Deterministic vivid colour for a class id (placeholder data only)."""
    hue = (cid * 0.61803398875) % 1.0
    r, g, b = colorsys.hsv_to_rgb(hue, 0.75, 0.95)
    return np.array([r * 255, g * 255, b * 255], dtype=np.float32)


def build_synthetic_dataset(root, n_train=60, n_val=20, n_test=20, size=IMG_SIZE, seed=SEED):
    """Create a tiny iSAID-*like* placeholder dataset (shapes on noisy backgrounds).

    This is NOT real remote-sensing data. It exists only so every cell of the
    notebook can be executed and debugged before the real iSAID subset is added.
    """
    rng = np.random.default_rng(seed)
    root = Path(root)
    img_dir, msk_dir, split_dir = root / "images", root / "masks", root / "splits"
    for d in (img_dir, msk_dir, split_dir):
        d.mkdir(parents=True, exist_ok=True)

    all_classes = BASE_CLASSES + UNSEEN_CLASSES
    yy, xx = np.ogrid[:size, :size]

    for split, n in (("train", n_train), ("val", n_val), ("test", n_test)):
        stems = []
        for i in range(n):
            stem = f"{split}_{i:03d}"
            # noisy, textured background
            img = rng.normal(110, 30, (size, size, 3)).clip(0, 255).astype(np.float32)
            # a few low-frequency colour blobs so the background is not uniform
            for _ in range(4):
                cx, cy = rng.integers(0, size, 2)
                rr = int(rng.integers(size // 8, size // 3))
                blob = ((xx - cx) ** 2 + (yy - cy) ** 2) <= rr * rr
                col = _class_color(int(rng.integers(1, 16)))
                img[blob] = img[blob] * 0.5 + col * 0.5
            mask = np.zeros((size, size), dtype=np.uint8)

            for _ in range(int(rng.integers(1, 4))):
                cname = str(all_classes[rng.integers(len(all_classes))])
                cid = CLASS_TO_ID[cname]
                color = _class_color(cid)
                x0, y0 = rng.integers(8, max(9, size - 96), 2)
                w, h = rng.integers(24, 90, 2)
                x1, y1 = min(size - 1, x0 + w), min(size - 1, y0 + h)
                if cname in ("storage tank", "roundabout", "swimming pool", "helicopter"):
                    region = ((xx - (x0 + x1) / 2) ** 2 / max((x1 - x0) / 2, 1) ** 2
                              + (yy - (y0 + y1) / 2) ** 2 / max((y1 - y0) / 2, 1) ** 2) <= 1
                elif cname == "plane":
                    region = ((xx >= x0) & (xx <= x1) & (yy >= (y0 + y1) / 2 - 6) & (yy <= (y0 + y1) / 2 + 6)) \
                             | ((yy >= y0) & (yy <= y1) & (xx >= (x0 + x1) / 2 - 6) & (xx <= (x0 + x1) / 2 + 6))
                else:  # elongated objects: ship, bridge, vehicles, courts, fields...
                    region = (xx >= x0) & (xx <= x1) & (yy >= y0) & (yy <= y1)
                region = np.asarray(region)
                img[region] = img[region] * 0.35 + color * 0.65
                mask[region] = cid

            Image.fromarray(img.astype(np.uint8)).save(img_dir / f"{stem}.png")
            Image.fromarray(mask).save(msk_dir / f"{stem}.png")
            stems.append(stem)
        (split_dir / f"{split}.txt").write_text("\n".join(stems) + "\n")

    return {"root": root, "images": img_dir, "masks": msk_dir}


# ----------------------------------------------------------------- locate ---
dataset_info = locate_dataset(DATA_ROOT_CANDIDATES)

if dataset_info is None:
    if ALLOW_SYNTHETIC_FALLBACK:
        print("=" * 78)
        print("!!  REAL iSAID-5i DATA NOT FOUND - GENERATING SYNTHETIC PLACEHOLDER DATA  !!")
        print("!!  The images below are coloured shapes, NOT satellite imagery.         !!")
        print("!!  Any metric computed on them is meaningless - do NOT report it.        !!")
        print("!!  Add the real iSAID subset (see the markdown above) and re-run.        !!")
        print("=" * 78)
        dataset_info = build_synthetic_dataset(DATA_ROOT)
        USING_SYNTHETIC = True
    else:
        raise FileNotFoundError(
            "No dataset found. Expected a folder containing 'images/' and 'masks/' subfolders.\n"
            f"Searched: {[str(p) for p in DATA_ROOT_CANDIDATES]}\n"
            "Download iSAID from https://captain-whu.github.io/iSAID/dataset.html, keep a subset\n"
            "of image/mask pairs, and place them in DATA_ROOT (or set ALLOW_SYNTHETIC_FALLBACK=True)."
        )
else:
    USING_SYNTHETIC = False

print("Dataset root :", dataset_info["root"])
print("Images dir   :", dataset_info["images"])
print("Masks dir    :", dataset_info["masks"])
n_files = len(list(dataset_info["images"].glob("*")))
print("Image files  :", n_files)
if n_files == 0:
    raise FileNotFoundError(f"No image files found in {dataset_info['images']}")


In [ ]:
import kagglehub

# Download the iSAID-5i dataset from Kaggle
kaggle_path = kagglehub.dataset_download("abinaya2704/isaid-5i-dataset")
print("Dataset downloaded to:", kaggle_path)

# Update configuration variables to point to the downloaded dataset
DATA_ROOT = Path(kaggle_path)
ALLOW_SYNTHETIC_FALLBACK = False

# Insert this path into the candidates list
DATA_ROOT_CANDIDATES.insert(0, DATA_ROOT)

In [ ]:
# =============================================================================
# 6c. IMAGE / MASK LOADING, RESIZING, NORMALISATION, Dataset AND DataLoader
# =============================================================================
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]
def load_mask(path, size=IMG_SIZE):
    """Load a semantic mask as an (H, W) int64 array of class ids, resized with
    nearest-neighbour interpolation so the class ids stay intact."""
    img = Image.open(path)
    if img.mode == "P":                       # palette PNG -> palette index = id
        arr = np.array(img).astype(np.int32)
    elif img.mode in ("L", "I", "I;16"):      # single channel id mask
        arr = np.array(img).astype(np.int32)
    else:                                     # RGB / RGBA colour mask
        rgb = np.array(img.convert("RGB")).astype(np.int32)
        grey = np.array_equal(rgb[..., 0], rgb[..., 1]) and np.array_equal(rgb[..., 0], rgb[..., 2])
        if grey:
            arr = rgb[..., 0]                 # ids stored as (id, id, id)
        elif CLASS_COLOR_MAP:
            arr = np.zeros(rgb.shape[:2], dtype=np.int32)
            for (r, g, b), cid in CLASS_COLOR_MAP.items():
                arr[(rgb == np.array([r, g, b])).all(axis=-1)] = cid
        else:
            raise ValueError(
                f"Mask '{path}' is a colour-coded RGB mask and CLASS_COLOR_MAP is empty.\n"
                "Set CLASS_COLOR_MAP = {(R,G,B): class_id} in the Configuration cell, "
                "or export single-channel id masks."
            )
    if arr.ndim == 3:
        arr = arr[..., 0]
    if arr.max() > 255:
        raise ValueError(f"Mask '{path}' has values up to {arr.max()}; expected class ids 0..15.")
    arr = arr.astype(np.uint8)
    if arr.shape[0] != size or arr.shape[1] != size:
        arr = np.array(Image.fromarray(arr).resize((size, size), Image.NEAREST))
    return arr.astype(np.int64)
def load_image(path, size=IMG_SIZE):
    """Load an RGB image resized to (size, size)."""
    img = Image.open(path).convert("RGB")
    if img.size != (size, size):
        img = img.resize((size, size), Image.BILINEAR)
    return img
def read_split_stems(info, split):
    """Return the list of file stems for a split, creating a deterministic
    70/15/15 split when splits/*.txt does not exist yet."""
    split_dir = Path(info["root"]) / "splits"
    split_dir.mkdir(exist_ok=True)
    f = split_dir / f"{split}.txt"
    if f.exists():
        return [l.strip() for l in f.read_text().splitlines() if l.strip()]
    # build once, then cache so later runs see the identical split
    all_stems = sorted(p.stem for p in Path(info["images"]).glob("*")
                       if p.suffix.lower() in IMG_EXTS)
    if not all_stems:
        return []
    rng = random.Random(SEED)
    rng.shuffle(all_stems)
    n = len(all_stems)
    n_train = max(1, int(n * 0.70))
    n_val = max(1, int(n * 0.15))
    parts = {
        "train": all_stems[:n_train],
        "val": all_stems[n_train:n_train + n_val],
        "test": all_stems[n_train + n_val:] or all_stems[-1:],
    }
    for name, stems in parts.items():
        (split_dir / f"{name}.txt").write_text("\n".join(stems) + "\n")
    return parts[split]
class ISaid5iDataset(Dataset):
    """iSAID-5i style dataset: RGB image + single-channel semantic mask.
    Parameters
    ----------
    info       : dict from locate_dataset() with keys root/images/masks
    split      : "train" | "val" | "test"
    allowed_ids: only images containing at least one of these class ids are kept
                 (train/val -> base classes, test -> unseen classes)
    max_images : cap on the number of kept images (manageable Colab subset)
    require_class: if True, only images containing at least one of allowed_ids
                 are kept (train/val -> base classes, test -> unseen classes)
    """
    def __init__(self, info, split, allowed_ids, img_size=IMG_SIZE,
                 max_images=None, max_scan=MAX_SCAN_IMAGES, require_class=True):
        self.images_dir = Path(info["images"])
        self.masks_dir = Path(info["masks"])
        self.img_size = img_size
        self.split = split
        self.allowed_ids = list(allowed_ids)
        stems = read_split_stems(info, split)
        self.samples = []          # list of (image_path, mask_path)
        self.present_ids = []      # list of [class ids present in that mask]
        scanned = 0
        for stem in stems:
            if max_images is not None and len(self.samples) >= max_images:
                break
            if scanned >= max_scan:
                break
            img_p = self._find_file(self.images_dir, stem)
            msk_p = self._find_file(self.masks_dir, stem)
            if img_p is None or msk_p is None:
                continue
            scanned += 1
            m = load_mask(msk_p, img_size)
            present = sorted({int(v) for v in np.unique(m) if v != 0})
            if require_class:
                keep = bool(set(present) & set(self.allowed_ids))
            else:
                keep = bool(present)
            if keep:
                self.samples.append((img_p, msk_p))
                self.present_ids.append(present)
        if len(self.samples) == 0:
            raise RuntimeError(
                f"Split '{split}' produced 0 usable samples after filtering for class ids "
                f"{self.allowed_ids}.\nImages: {self.images_dir}\nMasks: {self.masks_dir}\n"
                "Check that mask values really are class ids 0..15 (or set CLASS_COLOR_MAP)."
            )
    @staticmethod
    def _find_file(folder, stem):
        for ext in IMG_EXTS:
            p = folder / f"{stem}{ext}"
            if p.exists():
                return p
        return None
    def __len__(self):
        return len(self.samples)
    def __getitem__(self, i):
        img_p, msk_p = self.samples[i]
        img = load_image(img_p, self.img_size)
        mask_arr = load_mask(msk_p, self.img_size)
        # Remote Sensing Data Augmentation (flips preserve aerial geometry)
        if self.split == "train":
            if random.random() > 0.5:
                img = img.transpose(Image.FLIP_LEFT_RIGHT)
                mask_arr = np.ascontiguousarray(np.fliplr(mask_arr))
            if random.random() > 0.5:
                img = img.transpose(Image.FLIP_TOP_BOTTOM)
                mask_arr = np.ascontiguousarray(np.flipud(mask_arr))
        x = transforms.ToTensor()(img)                       # [0,1]
        x = transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)(x)
        mask = torch.from_numpy(mask_arr).long()
        return {"image": x, "mask": mask, "name": img_p.stem, "index": i}
def make_dataloaders(info):
    """Build train / val / test loaders.
    train + val are filtered to images containing BASE classes only;
    test   is filtered to images containing UNSEEN classes only.
    """
    train_set = ISaid5iDataset(info, "train", BASE_IDS, max_images=MAX_TRAIN_IMAGES)
    val_set = ISaid5iDataset(info, "val", BASE_IDS, max_images=MAX_VAL_IMAGES)
    test_set = ISaid5iDataset(info, "test", UNSEEN_IDS, max_images=None)
    train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=NUM_WORKERS, drop_last=False)
    val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False,
                            num_workers=NUM_WORKERS)
    test_loader = DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False,
                             num_workers=NUM_WORKERS)
    return train_set, val_set, test_set, train_loader, val_loader, test_loader
train_set, val_set, test_set, train_loader, val_loader, test_loader = make_dataloaders(dataset_info)
print(f"train images : {len(train_set):4d}   (base classes only)")
print(f"val   images : {len(val_set):4d}   (base classes only)")
print(f"test  images : {len(test_set):4d}   (unseen classes only)")
batch = next(iter(train_loader))
print("batch shapes :", batch["image"].shape, batch["mask"].shape,
      batch["mask"].dtype, "values:", torch.unique(batch["mask"]).tolist())

In [ ]:
# =============================================================================
# 7. DATASET VISUALIZATION - original image | ground truth mask
# =============================================================================
rng = np.random.default_rng(SEED)
_PALETTE = np.zeros((256, 3), dtype=np.uint8)
_PALETTE[0] = (30, 30, 30)
for i in range(1, 256):
    _PALETTE[i] = (colorsys.hsv_to_rgb((i * 0.61803398875) % 1.0, 0.7, 255)[0],
                   colorsys.hsv_to_rgb((i * 0.61803398875) % 1.0, 0.7, 255)[1],
                   colorsys.hsv_to_rgb((i * 0.61803398875) % 1.0, 0.7, 255)[2])


def colorize_mask(mask):
    """Map class ids to a fixed colour palette for display."""
    return _PALETTE[np.clip(mask.astype(np.int32), 0, 255)]


def denormalize(img_t):
    """Undo the ImageNet normalisation for display."""
    mean = torch.tensor(IMAGENET_MEAN, device=img_t.device).view(3, 1, 1)
    std = torch.tensor(IMAGENET_STD, device=img_t.device).view(3, 1, 1)
    return (img_t * std + mean).clamp(0, 1)


def show_dataset_examples(dataset, n=6, title="Dataset samples (image | ground truth mask)"):
    n = min(n, len(dataset))
    fig, axes = plt.subplots(2, n, figsize=(3.0 * n, 6.2))
    if n == 1:
        axes = axes.reshape(2, 1)
    for j in range(n):
        i = int(rng.integers(0, len(dataset)))
        s = dataset[i]
        axes[0, j].imshow(denormalize(s["image"]).permute(1, 2, 0).numpy())
        axes[0, j].set_title(s["name"], fontsize=8)
        axes[1, j].imshow(colorize_mask(s["mask"].numpy()))
        present = [ID_TO_CLASS.get(v, str(v)) for v in torch.unique(s["mask"]) if v > 0]
        axes[1, j].set_title("classes: " + (", ".join(present) or "-"), fontsize=7)
        for ax in axes[:, j]:
            ax.axis("off")
    axes[0, 0].set_ylabel("Image", fontsize=12)
    axes[1, 0].set_ylabel("Mask", fontsize=12)
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()


show_dataset_examples(train_set, n=6, title="Training samples (base classes)")
if len(test_set) > 0:
    show_dataset_examples(test_set, n=6, title="Test samples (unseen classes)")


## 8. Base / unseen class split

```text
BASE classes (10)  -> used for TRAINING + VALIDATION
UNSEEN classes ( 5) -> used ONLY for zero-shot testing, never for training
```

**How "unseen" is enforced in this notebook:**

1. `BASE_CLASSES` and `UNSEEN_CLASSES` are disjoint (asserted in the Configuration cell).
2. Train and validation loaders are built **only** from images filtered against `BASE_IDS`.
3. The target class of every training step is sampled with `build_binary_targets(...)`,
   which only ever receives `BASE_IDS` - so unseen class names/prompts never enter training.
4. Unseen class names are encoded by CLIP **only in the zero-shot testing section**.

As in the standard few-shot protocol, objects of an unseen class may still appear as ordinary
*background pixels* inside a training image - they are never used as a **target** class, exactly
like novel objects appearing in the background of few-shot episodes.

```text
Class                                Role
----------------------------------   ------------------------------
storage tank, baseball diamond,      base / training classes
tennis court, basketball court,
ground track field, soccer ball
field, small vehicle, helicopter,
roundabout, swimming pool

ship, plane, harbor, bridge,         unseen / zero-shot test classes
large vehicle                        (never used for training)
```


In [ ]:
# =============================================================================
# 8b. VERIFY THE SPLIT
# =============================================================================
assert set(BASE_CLASSES).isdisjoint(set(UNSEEN_CLASSES)), "class split must be disjoint"

# every training/validation target class must belong to BASE_IDS
for _cid in BASE_IDS:
    assert _cid not in UNSEEN_IDS

# which base / unseen classes actually occur in the data subsets we built?
def classes_present(dataset, ids):
    out = {}
    for cid in ids:
        n = sum(1 for p in dataset.present_ids if cid in p)
        out[ID_TO_CLASS[cid]] = n
    return out


print("Classes present in TRAIN images:", classes_present(train_set, BASE_IDS))
print("Classes present in VAL   images:", classes_present(val_set, BASE_IDS))
print("Classes present in TEST  images:", classes_present(test_set, UNSEEN_IDS))
print()
print(f"Unseen classes used for zero-shot test : {UNSEEN_CLASSES}")
print(f"Unseen classes used for training       : []   (never)")


## 9. Zero-shot prototype generation with CLIP

Because there is **no support image and no support mask** in the zero-shot setting, the initial
class prototype has to come from language:

```text
Original paper :  Support images + support masks  ->  visual prototypes  (masked average pooling)
Our zero-shot   :  Class names / text prompts     ->  CLIP text prototypes
```

Every class is described with remote-sensing prompt templates such as

```text
"a satellite image containing a {class_name}"
"a remote sensing image of a {class_name}"
"an aerial image of a {class_name}"
```

and encoded with the frozen CLIP text encoder. The embeddings of the multiple templates are
averaged and L2-normalised - these are the **initial semantic prototypes**. A small learnable
*text prototype adapter* (next sections) then moves them into the shared visual embedding space.


In [ ]:
# =============================================================================
# 9. CLIP TEXT PROTOTYPE GENERATION
# =============================================================================
CLIP_MODEL_ID = "openai/clip-vit-base-patch32"

clip_model = CLIPModel.from_pretrained(CLIP_MODEL_ID).to(DEVICE).eval()
clip_processor = CLIPProcessor.from_pretrained(CLIP_MODEL_ID)
for _p in clip_model.parameters():
    _p.requires_grad_(False)          # CLIP stays frozen, like the paper's frozen backbone

PROMPT_TEMPLATES = [
    "a satellite image containing a {}",
    "a remote sensing image of a {}",
    "an aerial image of a {}",
]


@torch.no_grad()
def _clip_text_features(texts):
    """Encode a list of prompts and return a plain (T, D) tensor.

    transformers 4.x returns a tensor from `get_text_features`, while
    transformers 5.x returns a model-output object whose pooled output is the
    projected text embedding - both are handled here.
    """
    inputs = clip_processor(text=texts, return_tensors="pt",
                            padding=True, truncation=True).to(DEVICE)
    feat = clip_model.get_text_features(**inputs)
    if torch.is_tensor(feat):
        return feat
    for attr in ("text_embeds", "pooler_output", "last_hidden_state"):
        val = getattr(feat, attr, None)
        if val is not None:
            if attr == "last_hidden_state":          # fall back to the first token
                val = val[:, 0]
            return val
    raise TypeError(f"Unexpected CLIP text output type: {type(feat)}")


@torch.no_grad()
def encode_class_names(class_names, templates=PROMPT_TEMPLATES):
    """Encode class names with the CLIP text encoder.

    Returns (N, EMBED_DIM) L2-normalised prototypes - the zero-shot replacement
    for the support-set prototypes of the original paper.
    """
    emb_list = []
    for name in class_names:
        pretty = name.replace("_", " ")
        texts = [t.format(pretty) for t in templates]
        feat = _clip_text_features(texts)
        feat = feat / feat.norm(dim=-1, keepdim=True)   # per-template unit vectors
        proto = feat.mean(dim=0)                        # average over templates
        proto = proto / proto.norm()                    # final unit prototype
        emb_list.append(proto)
    return torch.stack(emb_list, dim=0)


# prototypes for every class (base ones are used in training, unseen ones at test time)
ALL_CLASS_NAMES = BASE_CLASSES + UNSEEN_CLASSES
text_proto_all = encode_class_names(ALL_CLASS_NAMES).to("cpu")   # (15, 512)

CLASS_PROTO_INDEX = {name: i for i, name in enumerate(ALL_CLASS_NAMES)}

text_proto_base = text_proto_all[[CLASS_PROTO_INDEX[c] for c in BASE_CLASSES]].to(DEVICE)
text_proto_unseen = text_proto_all[[CLASS_PROTO_INDEX[c] for c in UNSEEN_CLASSES]].to(DEVICE)

print("CLIP text prototypes:", tuple(text_proto_all.shape))
print("base prototypes      :", tuple(text_proto_base.shape),
      "| unseen prototypes:", tuple(text_proto_unseen.shape))

# quick sanity check: similar classes should be closer than unrelated ones
sim = (text_proto_all @ text_proto_all.t()).cpu().numpy()
example = "ship" if "ship" in CLASS_PROTO_INDEX else BASE_CLASSES[0]
j = CLASS_PROTO_INDEX[example]
most_similar = sorted(range(15), key=lambda i: -sim[j, i])[:4]
print(f"Closest classes to '{example}':",
      [(ISAID_CLASSES[i], round(float(sim[j, i]), 3)) for i in most_similar])


## 10. Visual feature extractor - ResNet-50 (spatial feature maps)

The paper uses a **frozen, pretrained ResNet-50/101** backbone and keeps the *spatial* feature
map (a `C x H x W` tensor), not a single classification vector:

```text
Image (3 x 256 x 256)
   |
ResNet-50 stem + layer1 + layer2 + layer3      -> 1024 x 16 x 16   (stride 16)
ResNet-50 layer4                               -> 2048 x  8 x  8   (stride 32)
   |  (upsample layer4 to 16x16 and concatenate)
Spatial feature map F  (3072 x 16 x 16)
   |
1x1 convolution projection  (next section)
   |
Pixel embeddings (512 x 16 x 16)   -- one embedding per spatial location
```

The backbone is completely frozen (including BatchNorm statistics) so that only the lightweight
projection / prototype layers are trained - exactly the "frozen pretrained backbone" strategy
described in the paper's implementation details.


In [ ]:
# =============================================================================
# 10. ResNet-50 FEATURE EXTRACTOR (frozen, keeps the spatial feature map)
# =============================================================================
class ResNetFeatureExtractor(nn.Module):
    """Frozen pretrained ResNet-50 that outputs a spatial feature map.

    out = concat(layer3(x), upsample(layer4(x)))  ->  (B, 3072, H/16, W/16)
    """

    def __init__(self, pretrained=True):
        super().__init__()
        weights = ResNet50_Weights.IMAGENET1K_V2 if pretrained else None
        net = torchvision.models.resnet50(weights=weights)
        self.stem = nn.Sequential(net.conv1, net.bn1, net.relu, net.maxpool)
        self.layer1 = net.layer1
        self.layer2 = net.layer2
        self.layer3 = net.layer3      # stride 16, 1024 channels
        self.layer4 = net.layer4      # stride 32, 2048 channels
        self.out_channels = 1024 + 2048
        for p in self.parameters():
            p.requires_grad_(False)   # frozen backbone (paper: "pretrained backbone ... frozen")

    def forward(self, x):
        c1 = self.stem(x)
        c2 = self.layer1(c1)
        c3 = self.layer2(c2)
        f3 = self.layer3(c3)                                   # (B, 1024, h,  w)
        f4 = self.layer4(f3)                                   # (B, 2048, h/2, w/2)
        f4 = F.interpolate(f4, size=f3.shape[-2:], mode="bilinear", align_corners=False)
        return torch.cat([f3, f4], dim=1)                      # (B, 3072, h, w)


## 11. Projection / alignment layer

> **Important:** we never compute cosine similarity between *raw ResNet features* and *CLIP text
> embeddings* - they live in two different spaces. A **learnable projection** maps the visual
> feature map into the same `512`-dimensional space used by the CLIP text prototypes, so all
> similarities in the rest of the notebook are computed in one shared embedding space.

```text
ResNet feature (3072) --1x1 conv + GroupNorm + ReLU + 1x1 conv--> embedding (512), L2-normalised
CLIP text prototype (512) --learnable text adapter--> refined prototype (512), L2-normalised
```

This projection together with the text adapter are the main **trainable** parts of the network.


In [ ]:
# =============================================================================
# 11. PROJECTION / ALIGNMENT LAYERS
# =============================================================================
class ProjectionHead(nn.Module):
    """Maps ResNet features into the CLIP text embedding space (common space)."""

    def __init__(self, in_channels, embed_dim=EMBED_DIM):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_channels, 512, kernel_size=1),
            nn.GroupNorm(8, 512),
            nn.ReLU(inplace=True),
            nn.Conv2d(512, embed_dim, kernel_size=1),
        )

    def forward(self, f):
        z = self.net(f)                       # (B, 512, h, w)
        return F.normalize(z, dim=1)          # L2-normalise over the channel dim


class TextPrototypeAdapter(nn.Module):
    """Prototype refinement: a small residual MLP that moves CLIP text prototypes
    from the language space into the visual (projected) space."""

    def __init__(self, embed_dim=EMBED_DIM):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim, embed_dim),
            nn.ReLU(inplace=True),
            nn.Linear(embed_dim, embed_dim),
        )
        self.scale = nn.Parameter(torch.tensor(0.1))   # starts almost at raw CLIP space

    def forward(self, protos):
        return F.normalize(protos + self.scale * self.mlp(protos), dim=-1)


## 12. HPR-inspired hyperprototype module (simplified HPC)

The paper's **Adaptive Hyperprototype Representation (HPR)** = **Hyperprototype Clustering (HPC)**
+ **Guided Prototype Matching (GPM)**. Its purpose:

> Represent intra-class variation using **multiple** prototypes rather than relying on one.

HPC clusters the (masked) feature vectors with a SLIC/MaskSLIC-style superpixel procedure and
returns the cluster centroids = **hyperprototypes**. Here we implement a **simplified
HPC-inspired** version that keeps the same idea but uses a light k-means in the *joint
feature + spatial* space, mirroring the paper's distance

```text
D = sqrt( d_feature^2 + ( d_spatial / r )^2 )
```

The pixels that are clustered are either

* the most similar pixels to the text prototype (initial pseudo foreground), or
* the confident foreground pixels predicted by SSM (second round).

```python
# Simplified HPR-inspired hyperprototype representation
```

This is **not** an exact implementation of the paper's HPR - it is a simplified, easier to read
substitute with the same goal.


In [ ]:
# =============================================================================
# 12. HPR-INSPIRED HYPERPROTOTYPE MODULE  (simplified HPC)
# =============================================================================
# Simplified HPR-inspired hyperprototype representation


def kmeans_prototypes(feats, k, coords=None, iters=KMEANS_ITERS, spatial_weight=0.5):
    """Lightweight k-means used as the simplified hyperprototype clustering (HPC).

    feats  : (M, C) selected pixel embeddings (gradients allowed)
    coords : (M, 2) normalised (y, x) positions, adds the spatial term of Eq. (1)
    returns: (<= k, C) cluster centroids ("hyperprototypes")

    Cluster *assignments* are computed without gradient (like the paper's
    parameter-free clustering), while the centroids are computed with gradient
    so that the segmentation loss still trains the projection layer.
    """
    M, C = feats.shape
    k = max(1, min(k, M))
    f = feats.detach()
    f_aug = torch.cat([f, coords.detach() * spatial_weight], dim=1) if coords is not None else f

    # --- deterministic greedy initialisation (points that are far apart) ----
    centre_idx = [int(torch.argmax(f_aug.norm(dim=1)))]
    d = torch.cdist(f_aug, f_aug[centre_idx]).squeeze(1)
    while len(centre_idx) < k:
        idx = int(torch.argmax(d))
        centre_idx.append(idx)
        d = torch.minimum(d, torch.cdist(f_aug, f_aug[[idx]]).squeeze(1))
    centres = f_aug[centre_idx]

    # --- a few assignment / update iterations ------------------------------
    for _ in range(iters):
        assign = torch.cdist(f_aug, centres).argmin(dim=1)          # (M,) no-grad
        new_centres = []
        for j in range(k):
            m = assign == j
            new_centres.append(f_aug[m].mean(dim=0) if m.any() else centres[j])
        centres = torch.stack(new_centres)

    # --- final assignment (no grad), differentiable centroids -------------
    with torch.no_grad():
        d_final = torch.cdist(f_aug, centres)                       # (M, k)
        assign = d_final.argmin(dim=1)
        onehot = F.one_hot(assign, k).T.float()                     # (k, M)
        empty = onehot.sum(dim=1) == 0
        if empty.any():                                             # rescue empty clusters
            far = d_final.max(dim=0).indices
            for j in torch.nonzero(empty, as_tuple=False).squeeze(1):
                onehot[j] = 0.0
                onehot[j, far[j]] = 1.0
    counts = onehot.sum(dim=1, keepdim=True).clamp(min=1.0)
    centroids = (onehot @ feats) / counts                           # (k, C), grad flows
    return centroids


class SimplifiedHPR(nn.Module):
    """Simplified HPR / HPC: turn ONE text prototype into SEVERAL hyperprototypes
    by clustering the pixels that look like that class."""

    def __init__(self, num_hyperprotos=NUM_HYPERPROTOS, iters=KMEANS_ITERS, spatial_weight=0.5):
        super().__init__()
        self.k = num_hyperprotos
        self.iters = iters
        self.spatial_weight = spatial_weight

    def forward(self, pixel_embeds, base_proto, select_mask=None):
        # pixel_embeds : (HW, C) L2-normalised pixel embeddings
        # base_proto   : (C,)    initial (CLIP text) prototype
        # select_mask  : (HW,)   bool pixels to cluster (pseudo foreground), optional
        hw, c = pixel_embeds.shape
        k = min(self.k, hw)
        sim = pixel_embeds @ base_proto                              # (HW,)

        if select_mask is None:
            # pseudo foreground = the most similar 25% of all pixels
            n_sel = max(k, int(0.25 * hw))
            sel = torch.zeros(hw, dtype=torch.bool, device=pixel_embeds.device)
            sel[torch.topk(sim, n_sel).indices] = True
            select_mask = sel
        if int(select_mask.sum()) < k:
            sel = torch.zeros(hw, dtype=torch.bool, device=pixel_embeds.device)
            sel[torch.topk(sim, k).indices] = True                   # safety fallback
            select_mask = sel

        pts = pixel_embeds[select_mask]                              # (M, C)

        # 2-D positions of the selected pixels (feature map is square)
        side = max(int(round(math.sqrt(hw))), 1)
        idx = torch.nonzero(select_mask, as_tuple=False).squeeze(1)
        yy = (idx // side).float() / max(side - 1, 1)
        xx = (idx % side).float() / max(side - 1, 1)
        coords = torch.stack([yy, xx], dim=1)

        hyper = kmeans_prototypes(pts, k, coords=coords,
                                  iters=self.iters, spatial_weight=self.spatial_weight)
        return F.normalize(hyper, dim=1)                             # (<=k, C)


## 13. Guided Prototype Matching (GPM)

For every pixel the notebook computes the **cosine similarity** between the pixel embedding and
each class hyperprototype, then aggregates those similarities into a class score, exactly the
matching spirit of the paper's Eq. (4)-(5):

```text
pixel embedding
      |
      v
cosine similarity   ->   K hyperprototypes   (+ background prototype)
      |
      v
argmax  = guided map (which hyperprototype "owns" this pixel)
softmax-weighted mean = aggregated class score
      |
      v
per-pixel scores  ->  segmentation map
```


In [ ]:
# =============================================================================
# 13. GUIDED PROTOTYPE MATCHING (GPM)
# =============================================================================
def gpm_match(pixel_embeds, prototypes, temperature=1.0):
    """Guided Prototype Matching (simplified GPM, cf. paper Eq. 4-5).

    pixel_embeds : (HW, C) L2-normalised pixel embeddings
    prototypes   : (K,  C) L2-normalised class hyperprototypes

    returns
        score  : (HW,)   aggregated per-pixel class score
        assign : (HW,)   id of the best matching hyperprototype (guided map)
        sim    : (HW, K) raw cosine similarities (useful for visualisation)
    """
    sim = pixel_embeds @ prototypes.t()                 # (HW, K) cosine similarity
    weights = F.softmax(sim / temperature, dim=1)       # which hyperprototype matches best
    score = (weights * sim).sum(dim=1)                  # aggregated class score
    assign = sim.argmax(dim=1)
    return score, assign, sim


def scores_to_logits(fg_score, bg_score, scale=SCORE_SCALE):
    """Stack [background, foreground] scores -> (HW, 2) logits for CrossEntropyLoss."""
    return torch.stack([scale * bg_score, scale * fg_score], dim=-1)


## 14. Self-Supporting Matching (SSM-inspired refinement)

The paper's SSM says: *the prediction itself can support the prototype learning*
(Gestalt law - pixels of the same object are more similar to each other than to the support set).

```text
SSM-inspired self-supporting prototype refinement

1. initial prediction   (text prototypes matched against pixel embeddings)
2. select high-confidence foreground pixels  (and background pixels)
3. aggregate those reliable pixels  ->  query (self-support) prototype(s)
4. re-match  ->  refined prediction
```

Because there is no support set in the zero-shot setting, **all** prototypes come from the query
image itself at this stage - which is precisely the "query set guiding the query set" idea of the
paper (its Fig. 1 bottom half).


In [ ]:
# =============================================================================
# 14. SSM-INSPIRED SELF-SUPPORTING PROTOTYPE REFINEMENT
# =============================================================================
def confident_masks(logits, fg_thresh=FG_THRESH, bg_thresh=BG_THRESH):
    """Turn the initial prediction into high-confidence pseudo pixels.

    logits : (HW, 2) columns = [background, foreground]
    returns: (fg_confident, bg_confident, prob) boolean masks of shape (HW,)
    """
    prob = F.softmax(logits, dim=-1)
    pred = logits.argmax(dim=-1)
    fg_conf = (prob[:, 1] >= fg_thresh) & (pred == 1)
    bg_conf = (prob[:, 0] >= bg_thresh) & (pred == 0)
    return fg_conf, bg_conf, prob


def masked_average(pixel_embeds, mask):
    """Masked average pooling (MAP): aggregate selected pixels into ONE prototype."""
    if mask.dtype != torch.bool:
        mask = mask.bool()
    if int(mask.sum()) == 0:
        return None
    return F.normalize(pixel_embeds[mask].mean(dim=0), dim=0)


## 15. Adaptive Self-Supporting Background Prototype (ASBP)

Background is **not** just one fixed semantic class: it is cluttered and heterogeneous
(a ship query image may contain cars, water, docks... all "background" but visually unrelated).
The paper therefore builds an **adaptive, per-pixel background prototype** with an affinity
matrix (its Eq. 9-10):

```text
F_q,b  = confident background pixels            (M, C)
A      = F_q,b  x  F_q^T                        (M, HW)   affinity
W      = softmax(A, dim=0)                      normalised weights
P'_q,b = W^T x  F_q,b                           (HW, C)   one background prototype PER pixel
bg score at (x,y) = cosine( F_q(x,y) , P'_q,b(x,y) )
```

Difference between the two kinds of prototypes:

```text
Foreground semantic prototypes : one per CLASS, derived from the CLIP class name
                                 (several hyperprototypes per class via HPR)
Background prototype           : derived from the image ITSELF, from pixels that
                                 confidently look like "not the class" - adaptive,
                                 per-pixel, never from a class name
```


In [ ]:
# =============================================================================
# 15. ASBP - ADAPTIVE SELF-SUPPORTING BACKGROUND PROTOTYPE
# =============================================================================
def asbp_background(pixel_embeds, bg_mask, max_bg_pixels=512):
    """Adaptive Self-Supporting Background Prototype (cf. paper Eq. 9-10).

    pixel_embeds : (HW, C) L2-normalised pixel embeddings
    bg_mask      : (HW,)   bool, confident background pixels

    returns
        bg_proto_map : (HW, C) adaptive background prototype for every pixel
        bg_score     : (HW,)   per-pixel similarity to its own background prototype
    """
    hw, c = pixel_embeds.shape
    idx = torch.nonzero(bg_mask.bool(), as_tuple=False).squeeze(1)

    if idx.numel() == 0:                                   # fallback: global background
        proto = F.normalize(pixel_embeds.mean(dim=0), dim=0)
        return proto.unsqueeze(0).expand(hw, -1), (pixel_embeds @ proto)

    if idx.numel() > max_bg_pixels:                        # keep the affinity matrix small
        pick = torch.linspace(0, idx.numel() - 1, max_bg_pixels,
                              device=idx.device).long()
        idx = idx[pick]

    fb = pixel_embeds[idx]                                 # (M, C) background features
    aff = fb @ pixel_embeds.t()                            # A = (M, HW)   affinity matrix
    w = F.softmax(aff, dim=0)                              # softmax over first dimension
    bg_proto_map = w.t() @ fb                              # (HW, C) adaptive prototypes
    bg_proto_map = F.normalize(bg_proto_map, dim=1)
    bg_score = (pixel_embeds * bg_proto_map).sum(dim=1)
    return bg_proto_map, bg_score


def global_background(pixel_embeds, fg_score, frac=0.5):
    """Single global background prototype: mean of the least foreground-like pixels.
    Used as the simple (non-adaptive) background of the baseline model."""
    hw = pixel_embeds.shape[0]
    n = max(1, int(hw * frac))
    sel = torch.topk(fg_score, n, largest=False).indices
    mask = torch.zeros(hw, dtype=torch.bool, device=pixel_embeds.device)
    mask[sel] = True
    proto = F.normalize(pixel_embeds[mask].mean(dim=0), dim=0)
    return proto, (pixel_embeds @ proto)


## 16. Complete zero-shot model

```text
                ZERO-SHOT CLASS INFORMATION
                         |
                 Class Names / Prompts
                         |
                    CLIP Text Encoder  (frozen)
                         |
                  Text Prototypes
                         |
                 Prototype Refinement (learnable adapter)
                  / HPR-inspired module (hyperprototypes)
                         |
                         v

Query Image -> ResNet-50 -> Spatial Feature Map (3072 x 16 x 16)
                         |
                   Projection Layer  (learnable 1x1 convs)
                         |
                   Pixel Embeddings  (512 x 16 x 16)
                         |
            Guided Prototype Matching (cosine similarity)
                         |
             SSM / Self-Supporting Refinement
                         |
                 ASBP Background (adaptive)
                         |
                   Refine Prediction
                         |
                         v
                Zero-Shot Segmentation
```

Forward pass, per image:

1. `ResNet-50` (frozen) -> spatial feature map -> `ProjectionHead` -> pixel embeddings.
2. CLIP text prototype of the target class -> `TextPrototypeAdapter` -> base prototype.
3. **HPR**: cluster the most similar pixels -> `K` hyperprototypes (+ the base prototype).
4. **GPM**: cosine match pixels against the hyperprototypes -> initial prediction
   (background prototype = mean of the least foreground-like pixels).
5. **SSM**: keep high-confidence foreground/background pixels -> query prototypes ->
   fuse with the initial prototypes using learnable weights `lambda` (paper Eq. 11-12).
6. **ASBP**: build one adaptive background prototype per pixel from confident background
   pixels (or a single background mean if ASBP is switched off).
7. **Refine**: use the refined prediction as secondary pseudo labels, extract prototypes
   once more and produce the final logits.

Every module is controlled by the `MODULES` flags, which is what the ablation study toggles.


In [ ]:
# =============================================================================
# 16. COMPLETE ZERO-SHOT MODEL
# =============================================================================
class ZeroShotSSMP(nn.Module):
    """Simplified zero-shot adaptation of the paper's SSMP framework."""

    def __init__(self, modules=None, embed_dim=EMBED_DIM):
        super().__init__()
        self.backbone = ResNetFeatureExtractor(pretrained=True)      # frozen
        self.project = ProjectionHead(self.backbone.out_channels, embed_dim)
        self.text_adapter = TextPrototypeAdapter(embed_dim)          # prototype refinement
        self.hpr = SimplifiedHPR()                                   # hyperprototypes
        self.log_lambda = nn.Parameter(torch.zeros(2))               # fusion weights (Eq. 11-12)
        self.flags = dict(MODULES if modules is None else modules)

    # keep the frozen backbone in eval mode so BatchNorm statistics never change
    def train(self, mode=True):
        super().train(mode)
        self.backbone.eval()
        return self

    # ---------------------------------------------------------- embeddings --
    def extract_embeddings(self, images):
        """images (B,3,H,W) -> pixel embeddings (B, H*W, C) + feature map size."""
        with torch.no_grad():
            feats = self.backbone(images)                 # frozen: no graph needed
        z = self.project(feats)                           # learnable alignment
        b, c, h, w = z.shape
        embeds = z.permute(0, 2, 3, 1).reshape(b, h * w, c)
        return embeds, (h, w)

    # ------------------------------------------------------------ helpers ---
    @staticmethod
    def _fuse(set_a, set_b, lam):
        """Weighted fusion of two prototype sets (paper Eq. 11-12), padded to equal length."""
        k = max(set_a.shape[0], set_b.shape[0])
        if set_a.shape[0] < k:
            set_a = torch.cat([set_a, set_a[-1:].repeat(k - set_a.shape[0], 1)], dim=0)
        if set_b.shape[0] < k:
            set_b = torch.cat([set_b, set_b[-1:].repeat(k - set_b.shape[0], 1)], dim=0)
        return F.normalize(lam[0] * set_a + lam[1] * set_b, dim=1)

    def _hyperprototypes(self, emb, base, select_mask=None):
        """HPR: several hyperprototypes instead of one single prototype."""
        if self.flags["hpr"]:
            extra = self.hpr(emb, base, select_mask=select_mask)
            return torch.cat([base.unsqueeze(0), extra], dim=0)      # (K+1, C)
        if select_mask is None:
            return base.unsqueeze(0)                                  # (1, C)
        q = masked_average(emb, select_mask)
        return (q if q is not None else base).unsqueeze(0)

    def _background_score(self, emb, bg_conf, bg_proto0):
        """ASBP adaptive per-pixel background, else a single confident-background mean."""
        if self.flags["asbp"]:
            _, bg_score = asbp_background(emb, bg_conf)
            return bg_score
        bg_p = masked_average(emb, bg_conf)
        if bg_p is None:
            bg_p = bg_proto0                      # fall back to the global background prototype
        return emb @ bg_p

    # -------------------------------------------------------------- forward --
    def forward(self, images, text_protos, target_idx):
        """
        images     : (B, 3, H, W)
        text_protos: (N, C) raw CLIP text prototypes (base + unseen)
        target_idx : list of prototype row indices (one per image)
        """
        embeds, (fh, fw) = self.extract_embeddings(images)
        lam = F.softmax(self.log_lambda, dim=0)

        final_logits, init_logits, fg_scores, assigns, proto_sets = [], [], [], [], []

        for i in range(embeds.shape[0]):
            emb = embeds[i]
            base = self.text_adapter(text_protos[target_idx[i]])      # refined text prototype

            # ---- (1) HPR + GPM: initial prediction (paper's support stage) ----
            hyper = self._hyperprototypes(emb, base)                  # HPR
            fg0, assign0, _ = gpm_match(emb, hyper)                   # GPM
            bg_proto0, bg_score0 = global_background(emb, fg0)        # simple background
            logits0 = scores_to_logits(fg0, bg_score0)
            fg_conf, bg_conf, _ = confident_masks(logits0)            # SSM pseudo pixels
            cur_logits, cur_set = logits0, hyper

            # ---- (2) SSM: self-supporting query prototypes ---------------------
            if self.flags["ssm"]:
                q_fg = masked_average(emb, fg_conf)
                if q_fg is None:
                    q_fg = base
                q_set = self._hyperprototypes(emb, q_fg, select_mask=fg_conf)
                fused = self._fuse(hyper, q_set, lam)
                fg1, assign1, _ = gpm_match(emb, fused)
                bg_score1 = self._background_score(emb, bg_conf, bg_proto0)
                cur_logits = scores_to_logits(fg1, bg_score1)
                cur_set = fused
                fg_conf2, bg_conf2, _ = confident_masks(cur_logits)   # secondary pseudo labels
                fg_cur, asg_cur = fg1, assign1
            else:
                fg_conf2, bg_conf2 = fg_conf, bg_conf
                fg_cur, asg_cur = fg0, assign0

            # ---- (3) refine: one more pseudo-label / prototype round ------------
            if self.flags["refine"]:
                q_fg2 = masked_average(emb, fg_conf2)
                if q_fg2 is None:
                    q_fg2 = cur_set[0]
                set2 = self._hyperprototypes(emb, q_fg2, select_mask=fg_conf2)
                fused2 = self._fuse(cur_set, set2, lam) if self.flags["ssm"] else set2
                fg2, assign2, _ = gpm_match(emb, fused2)
                bg_score2 = self._background_score(emb, bg_conf2, bg_proto0)
                cur_logits = scores_to_logits(fg2, bg_score2)
                cur_set = fused2
                fg_cur, asg_cur = fg2, assign2

            final_logits.append(cur_logits)
            init_logits.append(logits0)
            fg_scores.append(fg_cur)
            assigns.append(asg_cur)
            proto_sets.append(cur_set)

        return {
            "logits": torch.stack(final_logits, 0),      # (B, HW, 2)  [bg, fg]
            "logits_init": torch.stack(init_logits, 0),  # (B, HW, 2)  initial prediction
            "fg_score": torch.stack(fg_scores, 0),       # (B, HW)
            "assign": torch.stack(assigns, 0),           # (B, HW) hyperprototype ids
            "embeds": embeds,                            # (B, HW, C)
            "prototypes": proto_sets,                    # list of (K, C)
            "feat_size": (fh, fw),
        }


model = ZeroShotSSMP(modules=MODULES).to(DEVICE)
model.eval()                                    # frozen backbone must not update BN stats
n_total = sum(p.numel() for p in model.parameters())
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters     : {n_total:,}")
print(f"Trainable parameters : {n_trainable:,}  (projection + text adapter + lambda)")

# smoke test on one batch
_batch = next(iter(train_loader))
_smoke_rows = [CLASS_PROTO_INDEX[BASE_CLASSES[0]]] * _batch["image"].shape[0]
with torch.no_grad():
    _out = model(_batch["image"].to(DEVICE), text_proto_all.to(DEVICE), _smoke_rows)
print("forward ok -> logits", tuple(_out["logits"].shape),
      "| embeds", tuple(_out["embeds"].shape),
      "| prototypes", tuple(_out["prototypes"][0].shape))


## 17. Training

Training uses **only base classes** (unseen class names/prompts never enter this loop):

* optimizer: **Adam** over the lightweight trainable parts only,
* loss: cross-entropy over the `[background, foreground]` cosine-score maps
  (the paper uses BCE on cosine maps - cross-entropy on 2 channels is the equivalent form),
  with a **foreground class weight** (`FG_LOSS_WEIGHT`, default 3.0) so the rare object pixels
  are not drowned out by the much larger background area (standard class-imbalance remedy),
* plus the pseudo/self-supporting loss with the paper's balancing coefficient `beta = 0.2`
  (`L = L_seg + 0.2 * L_pseudo`),
* frozen ResNet-50 and frozen CLIP,
* automatic GPU use, fixed random seed, checkpoint saving.


In [ ]:
# =============================================================================
# 17. TRAINING  (helpers + one epoch)
# =============================================================================
TEXT_PROTOS_DEV = text_proto_all.to(DEVICE)      # all class prototypes, single tensor
# class weights for the 2-channel [background, foreground] cross-entropy: without a
# foreground weight the model would quickly learn to predict "background everywhere"
CE_WEIGHTS = torch.tensor([1.0, FG_LOSS_WEIGHT], dtype=torch.float32, device=DEVICE)
BASE_CLASS_ROW = {CLASS_TO_ID[c]: i for i, c in enumerate(BASE_CLASSES)}
def build_binary_targets(masks, allowed_ids):
    """Choose one allowed (base) class per image and build a binary GT mask.
    Only `allowed_ids` (= BASE_IDS) are ever selected, so unseen classes can
    never become a training target.
    """
    allowed_ids = list(allowed_ids)
    targets = torch.zeros_like(masks)
    rows = []
    for i in range(masks.shape[0]):
        m = masks[i]
        present = [cid for cid in allowed_ids if bool((m == cid).any())]
        cid = random.choice(present) if present else random.choice(allowed_ids)
        rows.append(CLASS_PROTO_INDEX[ID_TO_CLASS[cid]])
        targets[i] = (m == cid).long()
    return targets, rows
def compute_segmentation_loss(out, targets):
    """L = L_seg + beta * L_pseudo   (paper Eq. 18, beta = 0.2)."""
    fh, fw = out["feat_size"]
    b = out["logits"].shape[0]
    tgt = F.interpolate(targets.float().unsqueeze(1), size=(fh, fw),
                        mode="nearest").squeeze(1).long()          # GT at feature resolution
    logits = out["logits"].permute(0, 2, 1).reshape(b, 2, fh, fw)
    loss = F.cross_entropy(logits, tgt, weight=CE_WEIGHTS)         # main segmentation loss
    if USE_PSEUDO_LOSS:
        init_logits = out["logits_init"].permute(0, 2, 1).reshape(b, 2, fh, fw)
        loss = loss + PSEUDO_LOSS_WEIGHT * F.cross_entropy(init_logits, tgt)
    return loss
def binary_iou(pred, gt):
    """IoU of two boolean tensors."""
    inter = torch.logical_and(pred, gt).sum().item()
    union = torch.logical_or(pred, gt).sum().item()
    return inter / max(union, 1)
trainable_params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.Adam(trainable_params, lr=LR, weight_decay=WEIGHT_DECAY)
print(f"Optimizer: Adam | lr={LR} | trainable tensors={len(trainable_params)}")
def train_one_epoch(model, loader, optimizer):
    """One pass over the base-class training set."""
    model.train()
    total, steps = 0.0, 0
    for batch in loader:
        images = batch["image"].to(DEVICE)
        masks = batch["mask"].to(DEVICE)
        targets, rows = build_binary_targets(masks, BASE_IDS)      # base classes only
        optimizer.zero_grad(set_to_none=True)
        out = model(images, TEXT_PROTOS_DEV, rows)
        loss = compute_segmentation_loss(out, targets)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(trainable_params, max_norm=1.0)
        optimizer.step()
        total += loss.item()
        steps += 1
    return total / max(steps, 1)

## 18. Validation

Validation runs on the **validation split (base classes)** and reports the mean IoU of the
binary foreground/background segmentation of the sampled base class - the same mIoU-style
metric that the paper uses, applied to base classes during training.


In [ ]:
# =============================================================================
# 18. VALIDATION
# =============================================================================
@torch.no_grad()
def validate(model, loader):
    """Validation loss + mean IoU on base classes (never unseen classes)."""
    model.eval()
    losses, ious = [], []
    for batch in loader:
        images = batch["image"].to(DEVICE)
        masks = batch["mask"].to(DEVICE)
        targets, rows = build_binary_targets(masks, BASE_IDS)
        out = model(images, TEXT_PROTOS_DEV, rows)
        losses.append(compute_segmentation_loss(out, targets).item())
        fh, fw = out["feat_size"]
        tgt = F.interpolate(targets.float().unsqueeze(1), size=(fh, fw),
                            mode="nearest").squeeze(1).long()
        pred = out["logits"].permute(0, 2, 1).reshape(-1, 2, fh, fw).argmax(dim=1)
        for j in range(pred.shape[0]):
            ious.append(binary_iou(pred[j] == 1, tgt[j] == 1))
    return float(np.mean(losses)), float(np.mean(ious))
def save_checkpoint(path, model, epoch, val_miou):
    torch.save({
        "projection": model.project.state_dict(),
        "text_adapter": model.text_adapter.state_dict(),
        "log_lambda": model.log_lambda.detach().cpu(),
        "flags": dict(model.flags),
        "epoch": epoch,
        "val_miou": val_miou,
    }, path)
def load_best_checkpoint(path=BEST_CKPT, model=model):
    ckpt = torch.load(path, map_location=DEVICE)
    model.project.load_state_dict(ckpt["projection"])
    model.text_adapter.load_state_dict(ckpt["text_adapter"])
    model.log_lambda.data.copy_(ckpt["log_lambda"].to(DEVICE))
    return ckpt
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=MIN_LR)
def fit(model, epochs=EPOCHS):
    """Full training loop with validation, learning rate scheduling, history and best-checkpoint saving."""
    history = {"train_loss": [], "val_loss": [], "val_miou": [], "lr": []}
    best_miou = -1.0
    for epoch in range(1, epochs + 1):
        t0 = time.time()
        cur_lr = optimizer.param_groups[0]['lr']
        train_loss = train_one_epoch(model, train_loader, optimizer)
        val_loss, val_miou = validate(model, val_loader)
        scheduler.step()
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["val_miou"].append(val_miou)
        history["lr"].append(cur_lr)
        save_checkpoint(CKPT_DIR / "last_ssmp_zeroshot.pt", model, epoch, val_miou)
        if val_miou > best_miou:
            best_miou = val_miou
            save_checkpoint(BEST_CKPT, model, epoch, val_miou)
        print(f"Epoch {epoch}/{epochs} | LR: {cur_lr:.6f}")
        print(f"Train Loss: {train_loss:.4f}")
        print(f"Validation mIoU: {val_miou:.4f}   (best {best_miou:.4f}, {time.time() - t0:.1f}s)")
        print("-" * 60)
    print(f"Training finished. Best checkpoint: {BEST_CKPT} (val mIoU {best_miou:.4f})")
    return history
print("Training setup ready.")

In [ ]:
# =============================================================================
# RUN TRAINING  (only base classes; unseen classes are untouched)
# =============================================================================
history = fit(model, epochs=EPOCHS)


In [ ]:
# =============================================================================
# RESTORE THE BEST CHECKPOINT
# =============================================================================
ckpt = load_best_checkpoint()
print(f"Loaded best checkpoint: epoch {ckpt['epoch']} | validation mIoU {ckpt['val_miou']:.4f}")
print("Checkpoint flags:", ckpt["flags"])

# re-evaluate the restored model on the validation split
val_loss, val_miou = validate(model, val_loader)
print(f"Restored model - val loss {val_loss:.4f} | val mIoU {val_miou:.4f}")


## 19. Zero-shot testing on unseen classes

```text
Unseen class names
       |
CLIP text encoder      ->  unseen semantic prototypes   (never seen in training)
       |
ResNet image features  ->  pixel embeddings             (projected to the shared space)
       |
HPR / hyperprototype matching                           (multiple prototypes per class)
       |
SSM-inspired refinement                                 (query supports itself)
       |
ASBP background handling                                (adaptive background prototype)
       |
final zero-shot segmentation
```

The model weights used here were trained **only** on base classes; the unseen class names are
used exclusively in this section.


In [ ]:
# =============================================================================
# 19. ZERO-SHOT INFERENCE + EVALUATION ON UNSEEN CLASSES
# =============================================================================
def logits_upsampled(out, size=IMG_SIZE):
    """(B, HW, 2) feature-resolution logits -> (B, 2, size, size)."""
    b = out["logits"].shape[0]
    fh, fw = out["feat_size"]
    logits = out["logits"].permute(0, 2, 1).reshape(b, 2, fh, fw)
    return F.interpolate(logits, size=(size, size), mode="bilinear", align_corners=False)


@torch.no_grad()
def predict_unseen(model, image_batch, class_name, modules=None):
    """Run the full zero-shot pipeline for one unseen class.

    image_batch : (B, 3, H, W) normalised images
    class_name  : unseen class name (its CLIP prompt is the prototype source)
    returns     : (B, H, W) boolean prediction + the raw model output
    """
    old_flags = dict(model.flags)
    if modules is not None:
        model.flags = dict(modules)
    model.eval()
    rows = [CLASS_PROTO_INDEX[class_name]] * image_batch.shape[0]
    out = model(image_batch, TEXT_PROTOS_DEV, rows)
    pred = (logits_upsampled(out).argmax(dim=1) == 1)
    model.flags = old_flags
    return pred, out


def evaluate_zero_shot(model, dataset, class_names=UNSEEN_CLASSES,
                       max_per_class=MAX_TEST_IMAGES, batch_size=4, modules=None):
    """Zero-shot evaluation on unseen classes (pixel-level accumulation).

    For every unseen class we use the test images in which that class actually
    occurs - the zero-shot analogue of sampling query episodes that contain the
    target class. Nothing here touches the training data.
    """
    used_flags = dict(modules) if modules is not None else dict(model.flags)
    orig_flags = dict(model.flags)
    if modules is not None:
        model.flags = dict(modules)
    model.eval()

    per_class = {}
    tp = fp = fn = tn = 0
    total_images = 0
    skipped = []

    with torch.no_grad():
        for cname in class_names:
            cid = CLASS_TO_ID[cname]
            idxs = [i for i, pres in enumerate(dataset.present_ids) if cid in pres]
            idxs = idxs[:max_per_class]
            if not idxs:
                skipped.append(cname)
                continue

            c_tp = c_fp = c_fn = 0
            for s in range(0, len(idxs), batch_size):
                chunk = idxs[s:s + batch_size]
                images = torch.stack([dataset[i]["image"] for i in chunk]).to(DEVICE)
                masks = torch.stack([dataset[i]["mask"] for i in chunk]).to(DEVICE)
                pred, _ = predict_unseen(model, images, cname, modules=modules)
                gt = (masks == cid).to(DEVICE)

                a = torch.logical_and(pred, gt)
                b = torch.logical_and(pred, torch.logical_not(gt))
                c = torch.logical_and(torch.logical_not(pred), gt)
                d = torch.logical_and(torch.logical_not(pred), torch.logical_not(gt))
                c_tp += int(a.sum().item()); c_fp += int(b.sum().item())
                c_fn += int(c.sum().item()); tn += int(d.sum().item())

            tp += c_tp; fp += c_fp; fn += c_fn
            total_images += len(idxs)
            per_class[cname] = {
                "iou": c_tp / max(c_tp + c_fp + c_fn, 1),
                "n_images": len(idxs),
                "tp": c_tp, "fp": c_fp, "fn": c_fn,
            }

    model.flags = orig_flags

    if not per_class:
        raise RuntimeError(
            "No test image contains any unseen class. Check the dataset split "
            f"(unseen classes: {class_names})."
        )

    ious = [v["iou"] for v in per_class.values()]
    fg_iou = tp / max(tp + fp + fn, 1)
    bg_iou = tn / max(tn + fp + fn, 1)
    return {
        "per_class": per_class,
        "mIoU": float(np.mean(ious)),
        "pixel_acc": float((tp + tn) / max(tp + fp + fn + tn, 1)),
        "dice": float(2 * tp / max(2 * tp + fp + fn, 1)),
        "fg_iou": float(fg_iou),
        "bg_iou": float(bg_iou),
        "fb_iou": float(0.5 * (fg_iou + bg_iou)),
        "n_images": total_images,
        "skipped": skipped,
        "flags": used_flags,
    }


# run the full zero-shot evaluation with all modules enabled
zero_shot_results = evaluate_zero_shot(model, test_set, modules=MODULES)


## 20. Evaluation metrics

`mIoU` is the primary metric of the paper. It is the mean over the unseen classes of the
per-class IoU (each class is evaluated as foreground vs. background, exactly as in few-shot
segmentation where every episode is binary).

Additional diagnostics reported here:

```text
Per-class IoU   : IoU of each unseen class (foreground only)
Foreground IoU  : IoU of foreground pixels pooled over all classes
Background IoU  : IoU of background pixels pooled over all classes
FB-IoU          : 0.5 * (Foreground IoU + Background IoU)      <- paper's 2nd metric
Pixel Accuracy  : (TP + TN) / all pixels
Dice / F1       : 2*TP / (2*TP + FP + FN)
```


In [ ]:
# =============================================================================
# 20. EVALUATION METRICS - formatted report
# =============================================================================
def print_metrics(results, title="ZERO-SHOT EVALUATION"):
    """Print the evaluation block requested by the assignment."""
    print("=" * 34)
    print(f"{title}")
    print("=" * 34)
    print(f"Images evaluated : {results['n_images']}")
    print(f"mIoU             : {results['mIoU'] * 100:.2f}%")
    print(f"Pixel Accuracy   : {results['pixel_acc'] * 100:.2f}%")
    print(f"Dice/F1          : {results['dice'] * 100:.2f}%")
    print(f"Foreground IoU   : {results['fg_iou'] * 100:.2f}%")
    print(f"Background IoU   : {results['bg_iou'] * 100:.2f}%")
    print(f"FB-IoU           : {results['fb_iou'] * 100:.2f}%")
    print("-" * 34)
    print("Per-class IoU:")
    for cname, v in sorted(results["per_class"].items()):
        print(f"  {cname:<22} {v['iou'] * 100:6.2f}%   ({v['n_images']} images)")
    if results.get("skipped"):
        print(f"  (no test images for: {results['skipped']})")
    print("Module flags:", results.get("flags"))
    print("=" * 34)
    return results


zero_shot_results = print_metrics(zero_shot_results)


## 21. Visualizations

1. Training curves (loss + validation mIoU)
2. Segmentation results (image / ground truth / zero-shot prediction / overlay)
3. Per-class IoU bar chart
4. Prototype similarity maps (incl. the hyperprototype guided map)
5. t-SNE of pixel embeddings vs. prototypes (inspired by the paper's Fig. 11)


In [ ]:
# =============================================================================
# 21.1 TRAINING CURVES
# =============================================================================
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(history["train_loss"], marker="o", label="train loss")
axes[0].plot(history["val_loss"], marker="s", label="val loss")
axes[0].set_xlabel("epoch"); axes[0].set_ylabel("loss")
axes[0].set_title("Training / validation loss"); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(history["val_miou"], marker="o", color="green", label="val mIoU")
axes[1].set_xlabel("epoch"); axes[1].set_ylabel("mIoU")
axes[1].set_title("Validation mIoU (base classes)"); axes[1].legend(); axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()


In [ ]:
# =============================================================================
# 21.2 SEGMENTATION RESULTS  (image | ground truth | prediction | overlay)
# =============================================================================
@torch.no_grad()
def show_segmentation_results(model, dataset, class_names=UNSEEN_CLASSES,
                              per_class=1, modules=MODULES):
    model.eval()
    rows = []
    for cname in class_names:
        cid = CLASS_TO_ID[cname]
        idxs = [i for i, pres in enumerate(dataset.present_ids) if cid in pres][:per_class]
        rows += [(i, cname, cid) for i in idxs]
    if not rows:
        print("No test images with unseen classes to visualise.")
        return

    fig, axes = plt.subplots(4, len(rows), figsize=(3.4 * len(rows), 12.5))
    if len(rows) == 1:
        axes = axes.reshape(4, 1)

    for col, (idx, cname, cid) in enumerate(rows):
        s = dataset[idx]
        img = (denormalize(s["image"]).permute(1, 2, 0).numpy() * 255).astype(np.uint8)
        gt_t = (s["mask"] == cid)
        gt = gt_t.numpy()
        pred_t, out = predict_unseen(model, s["image"].unsqueeze(0).to(DEVICE),
                                     cname, modules=modules)
        pred_t = pred_t[0].cpu()
        pred = pred_t.numpy()
        iou = binary_iou(pred_t, gt_t)

        axes[0, col].imshow(img)
        axes[0, col].set_title(f"{cname}\n{s['name']}", fontsize=9)

        gt_rgb = img.copy()
        gt_rgb[gt] = 0.45 * gt_rgb[gt] + 0.55 * np.array([0, 255, 0])   # green = GT
        axes[1, col].imshow(gt_rgb)
        axes[1, col].set_title("Ground truth", fontsize=9)

        pr_rgb = img.copy()
        pr_rgb[pred] = 0.45 * pr_rgb[pred] + 0.55 * np.array([255, 0, 0])  # red = prediction
        axes[2, col].imshow(pr_rgb)
        axes[2, col].set_title(f"Zero-shot prediction\nIoU {iou * 100:.1f}%",
                               fontsize=9)

        ov = img.copy()
        ov[gt & pred] = 0.35 * ov[gt & pred] + 0.65 * np.array([0, 255, 255])   # TP cyan
        ov[gt & ~pred] = 0.35 * ov[gt & ~pred] + 0.65 * np.array([255, 0, 0])   # FN red
        ov[~gt & pred] = 0.35 * ov[~gt & pred] + 0.65 * np.array([255, 255, 0]) # FP yellow
        axes[3, col].imshow(ov)
        axes[3, col].set_title("cyan=TP  red=FN  yellow=FP", fontsize=8)

        for r in range(4):
            axes[r, col].axis("off")

    axes[0, 0].set_ylabel("Image", fontsize=11)
    axes[1, 0].set_ylabel("Ground truth", fontsize=11)
    axes[2, 0].set_ylabel("Prediction", fontsize=11)
    axes[3, 0].set_ylabel("Overlay", fontsize=11)
    fig.suptitle("Zero-shot segmentation results (unseen classes)", fontsize=13)
    plt.tight_layout(); plt.show()


show_segmentation_results(model, test_set, per_class=1)


In [ ]:
# =============================================================================
# 21.3 PER-CLASS IoU BAR CHART
# =============================================================================
per_class = zero_shot_results["per_class"]
names = list(per_class.keys())
values = [per_class[n]["iou"] * 100 for n in names]

fig, ax = plt.subplots(figsize=(9, 4.5))
bars = ax.bar(names, values, color="steelblue", edgecolor="black")
for b, v in zip(bars, values):
    ax.text(b.get_x() + b.get_width() / 2, v + 0.5, f"{v:.1f}", ha="center", fontsize=9)
ax.axhline(zero_shot_results["mIoU"] * 100, color="red", ls="--",
           label=f"mIoU = {zero_shot_results['mIoU'] * 100:.2f}%")
ax.set_ylabel("IoU (%)")
ax.set_title("Per-class IoU - unseen (zero-shot) classes")
ax.set_ylim(0, min(100, max(values + [1]) * 1.25))
ax.legend(); plt.xticks(rotation=20, ha="right"); plt.tight_layout(); plt.show()


In [ ]:
# =============================================================================
# 21.4 PROTOTYPE SIMILARITY MAPS  (+ hyperprototype guided map)
# =============================================================================
@torch.no_grad()
def show_similarity_maps(model, dataset, class_names=UNSEEN_CLASSES, modules=MODULES):
    """Visualise the per-pixel similarity to the class hyperprototypes."""
    model.eval()
    chosen = []
    for cname in class_names:
        cid = CLASS_TO_ID[cname]
        idxs = [i for i, pres in enumerate(dataset.present_ids) if cid in pres][:1]
        chosen += [(i, cname, cid) for i in idxs]
    if not chosen:
        print("No images available for the similarity visualisation.")
        return

    n = len(chosen)
    fig, axes = plt.subplots(3, n, figsize=(3.6 * n, 10.5))
    if n == 1:
        axes = axes.reshape(3, 1)

    for col, (idx, cname, cid) in enumerate(chosen):
        s = dataset[idx]
        pred, out = predict_unseen(model, s["image"].unsqueeze(0).to(DEVICE),
                                   cname, modules=modules)
        fh, fw = out["feat_size"]
        gt = (s["mask"] == cid).numpy()
        img = denormalize(s["image"]).permute(1, 2, 0).numpy()

        sim = out["fg_score"][0].reshape(fh, fw).cpu().numpy()
        asg = out["assign"][0].reshape(fh, fw).cpu().numpy()

        axes[0, col].imshow(img)
        axes[0, col].set_title(f"{cname} - {s['name']}", fontsize=9)

        im1 = axes[1, col].imshow(F.interpolate(
            torch.from_numpy(sim)[None, None], size=(IMG_SIZE, IMG_SIZE),
            mode="bilinear", align_corners=False)[0, 0].numpy(), cmap="viridis")
        axes[1, col].set_title(f"similarity to '{cname}' prototypes", fontsize=9)
        fig.colorbar(im1, ax=axes[1, col], fraction=0.046)

        im2 = axes[2, col].imshow(F.interpolate(
            torch.from_numpy(asg).float()[None, None], size=(IMG_SIZE, IMG_SIZE),
            mode="nearest")[0, 0].numpy(), cmap="tab10")
        axes[2, col].set_title("GPM hyperprototype assignment", fontsize=9)
        fig.colorbar(im2, ax=axes[2, col], fraction=0.046)

        for r in range(3):
            axes[r, col].axis("off")

    axes[0, 0].set_ylabel("Image", fontsize=11)
    axes[1, 0].set_ylabel("Similarity", fontsize=11)
    axes[2, 0].set_ylabel("Assigned\nhyperproto", fontsize=11)
    fig.suptitle("Prototype similarity visualisation", fontsize=13)
    plt.tight_layout(); plt.show()


show_similarity_maps(model, test_set)


In [ ]:
# =============================================================================
# 21.5 t-SNE: pixel embeddings vs. class prototypes / hyperprototypes
# =============================================================================
@torch.no_grad()
def tsne_visualisation(model, dataset, class_names=UNSEEN_CLASSES,
                       max_classes=3, pixels_per_image=400, modules=MODULES):
    """Scatter plot of image pixels and prototypes in 2-D (cf. paper Fig. 11)."""
    model.eval()
    pts, pts_label, marker_type = [], [], []
    colour_names = ["background"]
    label_id = {"background": 0}

    def add_label(name):
        if name not in label_id:
            label_id[name] = len(colour_names)
            colour_names.append(name)
        return label_id[name]

    used = 0
    for cname in class_names:
        if used >= max_classes:
            break
        cid = CLASS_TO_ID[cname]
        idxs = [i for i, pres in enumerate(dataset.present_ids) if cid in pres][:1]
        if not idxs:
            continue
        used += 1
        s = dataset[idxs[0]]
        out = model(s["image"].unsqueeze(0).to(DEVICE),
                    TEXT_PROTOS_DEV, [CLASS_PROTO_INDEX[cname]])
        emb = out["embeds"][0].cpu()                      # (HW, 512)
        gt = (s["mask"] == cid)
        gt_small = F.interpolate(gt.float()[None, None], size=out["feat_size"],
                                 mode="nearest")[0, 0].bool().cpu()

        sel = np.random.default_rng(SEED).choice(emb.shape[0],
                                                 size=min(pixels_per_image, emb.shape[0]),
                                                 replace=False)
        for i in sel:
            pts.append(emb[i].numpy())
            pts_label.append(add_label(cname) if gt_small.view(-1)[i] else 0)
            marker_type.append("pixel")

        # the prototypes used for this image (text + HPR hyperprototypes)
        for p in out["prototypes"][0].cpu():
            pts.append(p.numpy())
            pts_label.append(add_label(f"{cname} (proto)"))
            marker_type.append("proto")

    if len(pts) < 10:
        print("Not enough data for t-SNE.")
        return

    X = np.stack(pts)
    perplexity = max(5, min(30, len(X) - 1))
    emb2d = TSNE(n_components=2, random_state=SEED, perplexity=perplexity,
                 init="pca", learning_rate="auto").fit_transform(X)

    fig, ax = plt.subplots(figsize=(9, 7))
    colours = plt.cm.tab20(np.linspace(0, 1, max(len(colour_names), 2)))
    for i, name in enumerate(colour_names):
        m = np.array(pts_label) == i
        if not m.any():
            continue
        is_proto = any(marker_type[j] == "proto" for j in np.where(m)[0][:1])
        ax.scatter(emb2d[m, 0], emb2d[m, 1], s=14 if is_proto else 6,
                   marker="*" if is_proto else "o", color=colours[i % len(colours)],
                   label=name, alpha=0.85 if is_proto else 0.5,
                   edgecolors="black" if is_proto else "none",
                   linewidths=0.6 if is_proto else 0)
    ax.set_title("t-SNE: pixel embeddings (dots) vs. prototypes (stars)")
    ax.legend(fontsize=8, loc="best", ncol=2)
    plt.tight_layout(); plt.show()


tsne_visualisation(model, test_set)


## 22. Ablation study

To isolate the contribution of each paper-inspired module we evaluate the **same trained model**
with different module combinations enabled:

```text
A: ResNet + CLIP prototype matching          (baseline, no extra module)
B: + HPR-inspired hyperprototypes
C: + SSM-inspired self-supporting refinement
D: + ASBP-inspired adaptive background
E: + refine (second pseudo-label round)      -> the full SSMP-inspired model
```

> These are **our own experimental numbers**, measured by running this notebook. They are **not**
> the numbers reported in the paper (the paper studies the *few-shot* task, this notebook studies
> the *zero-shot* task, so the values are not comparable). For speed the ablation uses
> `ABLATION_IMAGES_PER_CLASS` test images per unseen class; increase it for a fuller evaluation,
> or retrain per configuration (`fit(...)` after changing `MODULES`) for a stricter comparison.


In [ ]:
# =============================================================================
# 22. ABLATION STUDY
# =============================================================================
ABLATION_CONFIGS = [
    ("A: Baseline (ResNet + CLIP prototypes)", dict(hpr=False, ssm=False, asbp=False, refine=False)),
    ("B: + HPR",                               dict(hpr=True,  ssm=False, asbp=False, refine=False)),
    ("C: + SSM",                               dict(hpr=True,  ssm=True,  asbp=False, refine=False)),
    ("D: + ASBP",                              dict(hpr=True,  ssm=True,  asbp=True,  refine=False)),
    ("E: + refine (full model)",               dict(hpr=True,  ssm=True,  asbp=True,  refine=True)),
]

ablation_rows = []
for name, flags in ABLATION_CONFIGS:
    res = evaluate_zero_shot(model, test_set,
                             max_per_class=ABLATION_IMAGES_PER_CLASS, modules=flags)
    ablation_rows.append({
        "Model": name,
        "mIoU (%)": round(res["mIoU"] * 100, 2),
        "FB-IoU (%)": round(res["fb_iou"] * 100, 2),
        "Dice (%)": round(res["dice"] * 100, 2),
        "Pixel Acc (%)": round(res["pixel_acc"] * 100, 2),
    })
    print(f"{name:<42} mIoU = {res['mIoU'] * 100:6.2f}%")

ablation_df = pd.DataFrame(ablation_rows)
print()
print("=" * 40)
print("ABLATION SUMMARY (our own zero-shot results)")
print("=" * 40)
print(ablation_df.to_string(index=False))
print()
print("| Model | mIoU |")
print("|---|---:|")
for _, r in ablation_df.iterrows():
    print(f"| {r['Model']} | {r['mIoU (%)']:.2f}% |")

# restore the full configuration for the rest of the notebook
model.flags = dict(MODULES)


In [ ]:
# =============================================================================
# 23. FINAL RESULTS
# =============================================================================
print("=" * 60)
print("FINAL RESULTS - SSMP-inspired zero-shot semantic segmentation")
print("=" * 60)
if USING_SYNTHETIC:
    print("!! WARNING: these numbers were computed on the SYNTHETIC placeholder")
    print("!! dataset - they are NOT meaningful results. Add the real iSAID")
    print("!! subset and re-run before quoting any number.")
    print("-" * 60)

print(f"Dataset        : {'SYNTHETIC placeholder' if USING_SYNTHETIC else 'iSAID-5i subset'}")
print(f"Train images   : {len(train_set)}  (base classes: {BASE_CLASSES})")
print(f"Val images     : {len(val_set)}")
print(f"Test images    : {len(test_set)}  (unseen classes: {UNSEEN_CLASSES})")
print(f"Backbone       : frozen ImageNet-pretrained ResNet-50")
print(f"Prototype src  : frozen CLIP text encoder ({CLIP_MODEL_ID})")
print(f"Trained parts  : projection + text adapter + fusion weights "
      f"({n_trainable:,} parameters)")
print("-" * 60)
print_metrics(zero_shot_results, title="ZERO-SHOT EVALUATION (full model)")
print("-" * 60)
print("Ablation (mIoU):")
for _, r in ablation_df.iterrows():
    print(f"  {r['Model']:<42} {r['mIoU (%)']:6.2f}%")

# save everything so the numbers can be quoted later
summary = {
    "dataset": "synthetic-fallback" if USING_SYNTHETIC else "iSAID-5i subset",
    "base_classes": BASE_CLASSES,
    "unseen_classes": UNSEEN_CLASSES,
    "epochs": EPOCHS,
    "zero_shot": {k: v for k, v in zero_shot_results.items() if k != "per_class"},
    "per_class_iou": {k: v["iou"] for k, v in zero_shot_results["per_class"].items()},
    "ablation": ablation_rows,
    "history": history,
}
out_json = CKPT_DIR / "zero_shot_results.json"
out_json.write_text(json.dumps(summary, indent=2))
print(f"\nResults written to {out_json}")


## 24. Comparison with the original paper

| | Original paper (SSMP) | This notebook (SSMP-ZeroShot) |
|---|---|---|
| Task | Few-shot semantic segmentation | **Zero-shot** semantic segmentation |
| Class information | k support images + support masks | class names / text prompts (no support set) |
| Prototype source | masked average pooling of support features | frozen CLIP text encoder + learnable adapter |
| Backbone | frozen pretrained ResNet-50 / ResNet-101 | frozen ImageNet-pretrained ResNet-50 |
| HPR / HPC | SLIC + MaskSLIC-style superpixel clustering in feature space | simplified k-means in joint feature + spatial space |
| GPM | cosine similarity + argmax allocation (Eq. 4-5) | cosine similarity + argmax allocation + soft aggregation |
| SSM | pseudo mask -> query prototypes | same idea, one iteration, confidence thresholds |
| ASBP | affinity matrix of background features (Eq. 9-10) | same affinity formulation, capped background pixels |
| Prototype fusion | lambda-weighted fusion (Eq. 11-13) | learnable lambda-weighted fusion |
| Refine | second pseudo-label round | same, optional module |
| Losses | `L = L_seg + beta * L_pseudo`, beta = 0.2, BCE | `L = L_seg + beta * L_pseudo`, beta = 0.2, cross-entropy on 2 channels |
| Optimiser | SGD (momentum 0.9, wd 5e-4), 20 epochs | Adam, ~8 epochs (Colab-friendly) |
| Image size | 256 x 256 crops | 256 x 256 |
| Data | iSAID-5i (18,076 train / 6,363 val images) | a manageable **subset** of iSAID-5i |
| Metrics | mIoU + FB-IoU, 1000 test episodes | mIoU + per-class IoU + FB-IoU + Dice + pixel accuracy |
| Reported results (from the paper, not from this notebook) | one-shot mIoU **39.82%**, five-shot mIoU **50.52%** on iSAID-5i; ablation gains: SSM +2.93, refine +3.95, HPR +4.02 | printed by running this notebook |

**Numbers from the paper are quoted here only for context.** The zero-shot task of this notebook
is *harder* (no support images at all) and uses a subset of the data, so our metrics are **not
directly comparable** to the paper's few-shot numbers - and no number is hard-coded in this
notebook: everything you see above was produced by actually executing the cells.

### What stayed faithful to the paper

* prototype-based segmentation with **pixel-level cosine matching**
* **multiple hyperprototypes per class** (HPR) instead of a single prototype
* **self-supporting matching** (SSM): the prediction supports the prototype learning
* **adaptive self-supporting background prototype** (ASBP): background is not a fixed class
* prototype **refinement** rounds and **learnable fusion weights**
* frozen pretrained backbone, light trainable head, **mIoU / FB-IoU** evaluation


## 25. Limitations and future improvements

### Limitations

1. **Simplified HPC.** The paper clusters superpixels with a SLIC/MaskSLIC-style iterative seed
   growth in feature space; this notebook uses a small k-means with a spatial term. The goal
   (several hyperprototypes per class) is the same, the mechanism is simpler.
2. **No support set at all.** Zero-shot is a genuinely harder setting: the CLIP prompt must
   bridge the "language -> aerial image appearance" gap by itself, which limits the achievable
   mIoU compared to few-shot methods that see real support pixels.
3. **Subset of the data.** `MAX_TRAIN_IMAGES / MAX_TEST_IMAGES` are capped for Colab; results on
   the full iSAID-5i (18k+ images, 1000 evaluation episodes as in the paper) would be more
   reliable.
4. **Single fold.** Only one 5-class unseen fold is evaluated; the paper reports the average of
   three folds and five seeds.
5. **Ablation at inference time.** All module combinations reuse one trained model (fast). The
   paper trains each configuration separately.
6. **Backbone/CLIP gap.** ResNet-50 ImageNet features and CLIP text features are aligned only by
   a 1x1-convolution projection trained on base classes; a CLIP *image* encoder or a jointly
   trained adapter would align the spaces better.

### Possible improvements

* Evaluate all three iSAID-5i folds and average the mIoU.
* Train each ablation configuration from scratch (re-run `fit()` with different `MODULES`).
* Add more prompt templates / prompt tuning (CoOp-style) for the unseen classes.
* Replace k-means HPC with the paper's MaskSLIC seed-growth clustering.
* Use multi-scale features (layer2 + layer3 + layer4) for finer boundaries.
* Add a CRF or a light decoder to sharpen the 16x16 predictions up to 256x256.
* Extend to **generalised** zero-shot segmentation (base + unseen classes together).

---

## Final note on the relationship to the paper

```text
Original paper (few-shot)              This notebook (zero-shot)
--------------------------            -----------------------------
Support images + masks                Class names / text prompts
      |                                     |
      v                                     v
visual prototypes                      CLIP semantic prototypes
      |                                     |
      +--> HPR (HPC + GPM)             +--> HPR-inspired hyperprototypes
      |                                     |
      +--> SSM (pseudo labels)         +--> SSM-inspired refinement
      |                                     |
      +--> ASBP (background)           +--> ASBP-inspired background
      |                                     |
      v                                     v
segmentation (mIoU)                   segmentation (mIoU)
```

This notebook is a **simplified zero-shot adaptation inspired by the paper's SSMP framework**,
created as a project recreation. It is **not** an exact reproduction of the authors' original
few-shot SSMP implementation, and it is not a generic CLIP segmentation pipeline - every module
of the paper (HPR, HPC, GPM, SSM, ASBP, prototype refinement, pixel-level similarity matching,
mIoU evaluation) is present and clearly labelled in the code.
